**Sample Data**
**races_df (streamed first)**
**race_id	race_name	ingestion_date**
- 1	Monaco	2024-06-01 10:00:00
- 2	Silverstone	2024-06-01 10:05:00<br>
**qualifying_df (streamed next)**<br>
**race_id	driver_id	ingestion_date**<br>
1	101	2024-06-01 10:10:00<br>
2	102	2024-06-01 10:40:00<br>
2	103	2024-06-01 10:50:00<br>

**Watermark Setup**
Both DataFrames have:
This means Spark will keep state for each **row for up to 30 minutes after its ingestion_date**.

**How the Join Works**
Join Condition Explained
- **race_id equality:** Only rows with matching race_id are considered.
- **Time window:** For each qualifying_df row, Spark looks for races_df rows where
- **races_df.ingestion_date is between
qualifying_df.ingestion_date and qualifying_df.ingestion_date + 30 minutes.**

`%python
qualifying_df["race_id"] == races_df["race_id"],
races_df["ingestion_date"].between(
    qualifying_df["ingestion_date"],
    qualifying_df["ingestion_date"] + expr("interval 30 minutes")
)
`

Step-by-step Example
- 1.First, races_df rows arrive:
- Spark stores:
race_id=1, ingestion_date=10:00<br>
race_id=2, ingestion_date=10:05<br>
2.Next, qualifying_df rows arrive:<br>
Row 1:<br>
race_id=1, driver_id=101, ingestion_date=10:10<br>
Join condition<br>

Find races_df rows with race_id=1 and<br>
races_df.ingestion_date between 10:10 and 10:40<br>
races_df row:<br>

race_id=1, ingestion_date=10:00 (Does not match, because 10:00 < 10:10)<br>
Result:

- No match for this qualifying row.
- Row 2:
- race_id=2, driver_id=102, ingestion_date=10:40
- Join condition:

Find races_df rows with race_id=2 and<br>
races_df.ingestion_date between 10:40 and 11:10<br>
races_df row:

race_id=2, ingestion_date=10:05 (Does not match, because 10:05 < 10:40)
Result:

No match for this qualifying row.<br>
Row 3:<br>
race_id=2, driver_id=103, ingestion_date=10:50
Join condition:<br>

Find races_df rows with race_id=2 and<br>
races_df.ingestion_date between 10:50 and 11:20<br>
races_df row:<br>

race_id=2, ingestion_date=10:05 (Does not match, because 10:05 < 10:50)<br>
Result:

No match for this qualifying row.<br>
What if the join condition was reversed?<br>
If you had:<br>

Then for each races_df row, you’d look for qualifying_df rows whose ingestion_date is within 30 minutes after the race’s ingestion_date.<br>
`%python
qualifying_df["ingestion_date"].between(
    races_df["ingestion_date"],
    races_df["ingestion_date"] + expr("interval 30 minutes")
)`

Example:

races_df: race_id=1, ingestion_date=10:00<br>
qualifying_df: race_id=1, driver_id=101, ingestion_date=10:10
Check: Is 10:10 between 10:00 and 10:30?<br>
Yes! So this would match.


In [0]:
class Gold_race_results():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path="streaming_project/gold"

    def __init__(self,table,drivers_df,constructors_df,circuits_df):
         self.table=table
         self.drivers_df=drivers_df
         self.constructors_df=constructors_df
         self.circuits_df=circuits_df

    def max_watermark_value(self):
        from pyspark.sql.functions import max,col
        #fetching max_ingestion_date from gold layer table race_results
        if spark.catalog.tableExists("streaming_project.gold.race_results"):
            results_max_ingestion_date =spark.read.table('streaming_project.gold.race_results').agg(max(col('results_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if results_max_ingestion_date is None:
                results_max_ingestion_date='1900-01-01 00:00:00'
        else:
            results_max_ingestion_date='1900-01-01 00:00:00'
        print(f"results_max_ingestion_date:{results_max_ingestion_date}") #printing last water mark ingetsion timestamp

        return results_max_ingestion_date
            
    def read_input(self,list_max_ingest):
        from pyspark.sql.functions import max,col,expr,count
        results_max_ingestion_date=list_max_ingest
    
        #fetching incremental result data 
        incr_results_df= (spark.read.table('streaming_project.silver.results')
                        .filter(col('results_ingestion_date')>results_max_ingestion_date)
                            )
        
        #Printing incremental result data records count
        print("incr_results_df")
        display(incr_results_df.select(count(col('race_id'))))
       
        #fetching distinct race_id from incremental result data and printing count of records
        incr_results_df_list= incr_results_df.select(col("race_id").alias("result_race_id")).distinct()
        print("incr_results_df_list")
        display(incr_results_df_list.select(count(col('result_race_id'))))
        
        #fetching only requeried data from race table which is required for joining incremental result data and printing the count of records
        races_df=spark.read.table('streaming_project.silver.races')
        incr_race_df=races_df.join(incr_results_df_list,races_df["race_id"]==incr_results_df_list['result_race_id'],'inner').drop(col('result_race_id'))
        print("incr_race_df")
        display(incr_race_df.select(count(col('race_id'))))

        # drivers_df=spark.read.table('streaming_project.silver.drivers')
        # constructors_df=spark.read.table('streaming_project.silver.constructors')
        # circuits_df=spark.read.table('streaming_project.silver.circuits')
        #passing all requeried tables for join as list
        read_df_list=[incr_race_df,incr_results_df,self.drivers_df,self.constructors_df,self.circuits_df]
        return read_df_list
    
    def apply_transformations(self,read_df_list):
        from pyspark.sql.functions import round,col,broadcast,expr,count
        races_df = read_df_list[0]
        result_df = read_df_list[1]
        drivers_df=read_df_list[2]
        constructors_df=read_df_list[3]
        circuits_df=read_df_list[4]

        #performing join operation only on  incremental result data with all other tables
        race_results_df= (result_df.join(races_df,
           [result_df["race_id"] == races_df["race_id"]],'inner')
                .join(drivers_df,["driver_id" ],'inner')
                .join(constructors_df,["constructor_id"],'inner')
                .join(circuits_df,["circuit_id"],'inner')
                .selectExpr("race_year","race_name","circuit_name" ,"circuit_country","circuit_location","latitude","longitude","driver_name","driver_nationality", "constructor_team","constructor_nationality","grid","result_position","result_position_order","result_position_text","result_points","laps as result_laps","time as  result_time","fastest_lap as result_fastest_lap","fastest_lap_rank  as result_fastest_lap_rank","fastest_lap_time as result_fastest_lap_time","fastest_lap_speed as result_fastest_lap_speed","results_ingestion_date"
                                   )
                        )
        #displaying resulted data records count
        display(race_results_df.select(count("*")))
        return race_results_df

        
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approach using append mode
        (apply_tran_df.write.partitionBy("race_year","race_name")
         .mode("append")
         .saveAsTable(f"streaming_project.gold.{self.table}"))
        print("Data write into gold race_results table is Done")
    
    def process(self):
        print("Started gold-ingestion-race_results  is running....")
        list_max_ingest=self.max_watermark_value() # return latest water mark value
        read_df_list=self.read_input(list_max_ingest) #return list of all required tables
        apply_tran_df=self.apply_transformations(read_df_list) #returns joined table
        self.write_output(apply_tran_df) #write data into gold table
        return list_max_ingest
    


In [0]:
class Gold_season_summary():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path="streaming_project/gold"

    def __init__(self,table,max_ingestion_date):
         self.table=table
         self.max_ingestion_date=max_ingestion_date  #latest water mark value for race_results table from Gold_race_results class

    def read_input(self):
        from pyspark.sql.functions import max,col,expr,count
        race_year_list=list()
        #print the latest water mark value
        print(f"max_ingestion_date:{self.max_ingestion_date}")

        #fetching incremental race_results data from gold table race_results
        if spark.catalog.tableExists("streaming_project.gold.race_results"):
           Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
                                  .filter(col('results_ingestion_date')>self.max_ingestion_date))
           
           #fetching distinct race_year from incremental race_results data
           Incr_race_results_df_list = (Incr_race_results_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
           #fetching incremental race_results data and printing count of records
           print("season_summary:Incr_race_results_df batch count")
           display(Incr_race_results_df.select(count('*')))
           
           #listing of distinct race_years and printing the list
           race_year_list=[r.race_year for r in Incr_race_results_df_list]
           
        print(f"race_year_list:{race_year_list}")
        return race_year_list
    
    def apply_transformations(self,race_year_list):
        from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,count,min,when
        from pyspark.sql.window import Window
        spec_window=Window.partitionBy("race_year").orderBy(col('total_points').desc())
        
        #fetching only required data from race_results table which is required for aggregation and printing the count of records
        Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
                               .filter(col('race_year').isin(race_year_list))
                               )
        print("season_summary:Incr_race_results_df original count")
        display(Incr_race_results_df.select(count('*')))
        
        #aggregating data as per the requirements and printing out sample year data for clarification
        season_summary_df= (Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
                        .agg(sum(col("result_points")).alias("total_points"),
                        count(col("race_name")).alias("grand_prix_races"),
                        expr("count(case when result_position_order= 1 then result_position_order end)").alias("wins"),
                        min(col("result_position_order")).alias('Best_race_results_position'),
                    count(when(col("result_position") != 0,col("result_position"))).alias("No_of_race_finshes"),
                    count(when(col("result_position") == 0,col("result_position"))).alias("race_does_not_finshes"),
                    count(when(col("result_fastest_lap_rank") == 1, col("result_fastest_lap_rank"))).alias("fastest_laps"),
                    round(avg(col("result_position_order")),2).alias("Avg_race_position")          
                    ).withColumn("position",dense_rank().over(spec_window))
                    .select(col("race_year"),col("driver_name"),col("position"),col("total_points"),
                            col("grand_prix_races"),col("wins"),col("Best_race_results_position"),col("No_of_race_finshes"),col("race_does_not_finshes"),col("fastest_laps"),col("Avg_race_position"))
                         )
        display(season_summary_df.filter((col("race_year")==2018)))
        return season_summary_df
    
    def write_output(self,apply_tran_df):
         # writing those data into gold layer table by partitioning according to filter approache using dynamic partitionOverwriteMode as true with overwritte mode
        (apply_tran_df.write
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("race_year").mode("overwrite").saveAsTable(f"streaming_project.gold.{self.table}"))
        print("Data write into gold season_summary table is Done")
    
    def process(self):
        print("Started gold-ingestion-season_summary in runing....")
        race_year_list=self.read_input() #return list of distinct race_years
        apply_tran_df=self.apply_transformations(race_year_list) #return aggregated data
        self.write_output(apply_tran_df)#write data into gold table                    
        

In [0]:
class Gold_champion_list():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path="streaming_project/gold"

    def __init__(self,table,max_ingestion_date):
         self.table=table
         self.max_ingestion_date=max_ingestion_date #latest water mark value for race_results table from Gold_race_results class

    def read_input(self):
        from pyspark.sql.functions import max,col,expr,count
        race_year_list=list()
        #print the latest water mark value
        print(f"max_ingestion_date:{self.max_ingestion_date}")

        #fetching incremental race_results data from gold table race_results
        if spark.catalog.tableExists("streaming_project.gold.race_results"):
            Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
                                  .filter(col('results_ingestion_date')>self.max_ingestion_date))
            
            #fetching distinct race_year from incremental race_results data
            Incr_race_results_df_list =(Incr_race_results_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
            
            #fetching incremental race_results data and printing count of records
            print("champion_list:Incr_race_results_df batch count")
            display(Incr_race_results_df.select(count('*')))
            race_year_list=[r.race_year for r in Incr_race_results_df_list]
        #listing of distinct race_years and printing the list
        print(f"race_year_list:{race_year_list}")
        return race_year_list
    
    def apply_transformations(self,race_year_list):
        from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,when,concat,lit,count
        from pyspark.sql.window import Window
        spec_window=Window.partitionBy("race_year").orderBy(col('total_points').desc())

        #fetching only required data from race_results table which is required for aggregation and printing the count of records
        Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
                               .filter(col('race_year').isin(race_year_list))
                               )
        print("champion_list:Incr_race_results_df original count")
        display(Incr_race_results_df.select(count('*')))

        #aggregating data as per the requirements and printing out sample year data for clarification
        champions_df=(Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
                        .agg(sum(col("result_points")).alias("total_points"))
                        .withColumn("position",dense_rank().over(spec_window))
                        .withColumn("winning_position",when(col("position")==1,"Champion").otherwise(concat(lit("P_"),col('position'))))
                        .select(col("race_year"),col("driver_name"),col("total_points"),col("position"),col("winning_position"))
                    )
        display(champions_df.filter(col("driver_name")=="Lewis Hamilton"))
        return champions_df
    
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approache using dynamic partitionOverwriteMode as true with overwritte mode
        (apply_tran_df.write.option("partitionOverwriteMode", "dynamic")
         .partitionBy("race_year").mode("overwrite")
         .saveAsTable(f"streaming_project.gold.{self.table}"))
        print("Data write into gold champion_list table is Done")

    def process(self):
        print("Started gold-ingestion-champion_list in runing....")
        race_year_list=self.read_input()  #return list of distinct race_years
        apply_tran_df=self.apply_transformations(race_year_list) #return aggregated data
        self.write_output(apply_tran_df)#write data into gold table 


In [0]:
class Gold_driver_scd_details():
    main_path = "/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path = "streaming_project/gold"

    def __init__(self, table, max_ingestion_date):
        self.table = table
        self.max_ingestion_date = max_ingestion_date  # latest water mark value for race_results table from Gold_race_results class

    def read_input(self):
        from pyspark.sql.functions import max, col, expr, count
        race_year_list = list()
        print(f"max_ingestion_date:{self.max_ingestion_date}")

        if spark.catalog.tableExists("streaming_project.gold.race_results"):
            Incr_race_results_df = (
                spark.read.table('streaming_project.gold.race_results')
                .filter(col('results_ingestion_date') > self.max_ingestion_date)
                )
            Incr_race_results_distinct_df=(Incr_race_results_df.selectExpr("driver_name", "driver_nationality", "constructor_team", "race_year").distinct())
            print("champion_list:Incr_race_results_df batch count")
            display(Incr_race_results_distinct_df.select(count('*')))
        return Incr_race_results_distinct_df

    def apply_transformations(self, Incr_race_results_distinct_df):
      from pyspark.sql.functions import lit,col,dense_rank,max,row_number
      from pyspark.sql.window import Window

      if spark.catalog.tableExists("streaming_project.gold.driver_scd_details"):
        records_count=spark.sql("select count(*) as count from streaming_project.gold.driver_scd_details").collect()[0]['count']
        Incr_race_results_distinct_rank_df=Incr_race_results_distinct_df.withColumn('rank',row_number().over(Window.partitionBy(col("driver_name")).orderBy(col("race_year")))).orderBy(col('driver_name'),col('race_year')).select('driver_name','driver_nationality','constructor_team','race_year','rank')

        print("intial checkup")
        display(Incr_race_results_distinct_rank_df.filter(col('driver_name')=='Larry Perkins'))

        if records_count==0:
          (Incr_race_results_distinct_rank_df
          .withColumn('driver_scd_end_year',lit(None))
          .withColumn('driver_scd_status',lit(True))
          .filter(col('rank')==1)
          .selectExpr(
            "driver_name", "driver_nationality", "constructor_team", "race_year as driver_scd_start_year","driver_scd_end_year","driver_scd_status"
             ).createOrReplaceTempView("initial_driver_scd_details"))
          # printing intial loading data
          print("initial_driver_scd_details")
          display(spark.sql('''SELECT * FROM initial_driver_scd_details'''))

          print(f"duplicated data if any in initial_driver_scd_details")
          display(spark.sql('''
                            SELECT driver_name,count(*) as count FROM initial_driver_scd_details
                            group by driver_name
                            having count(*)>1
                            '''))
          spark.sql('''
            INSERT INTO streaming_project.gold.driver_scd_details
            SELECT * FROM initial_driver_scd_details
          ''')
        # max_rank value from Incr_race_results_distinct_df and printing it
        max_rank=(Incr_race_results_distinct_rank_df
         .agg(max(col("rank")).alias('max_rank'))).collect()[0]['max_rank']
        print(f"max_rank:{max_rank}")

        for r in range(2,int(max_rank)+1):
          batch_df=(Incr_race_results_distinct_rank_df.filter(col('rank')==r).orderBy(col('driver_name'),col('race_year'))
          .selectExpr("driver_name", "driver_nationality", "constructor_team", "race_year"))
          batch_df.createOrReplaceTempView("updated_driver_scd_details")
          
          print(f"{r}:batch_df")
          display(spark.sql('''SELECT * FROM updated_driver_scd_details'''))

          print(f"duplicated data if any in updated_driver_scd_details")
          display(spark.sql('''
                            SELECT driver_name,count(*) as count FROM updated_driver_scd_details
                            group by driver_name
                            having count(*)>1
                            '''))
          # SCD Type 2 Implementation
          spark.sql('''
              MERGE INTO streaming_project.gold.driver_scd_details t
              USING updated_driver_scd_details u
              ON t.driver_name = u.driver_name
                AND t.constructor_team != u.constructor_team
                AND t.driver_scd_status = TRUE
              WHEN MATCHED THEN UPDATE SET
                  t.driver_scd_end_year = u.race_year-1,
                  t.driver_scd_status = FALSE
              WHEN NOT MATCHED THEN INSERT (
                  driver_name,driver_nationality,constructor_team,driver_scd_start_year,driver_scd_end_year,driver_scd_status
              ) VALUES (
                  u.driver_name,u.driver_nationality,u.constructor_team,u.race_year,NULL,TRUE)
          ''')
          print("driver_scd_details with out appending new modified records")
          display(spark.sql('''SELECT * FROM streaming_project.gold.driver_scd_details'''))

          df1 = spark.sql("select * from streaming_project.gold.driver_scd_details where driver_scd_status=FALSE")

          
          insert_df = (
              batch_df.alias("src")
              .join(df1.alias("tgt"), ["driver_name",col('src.race_year')==col('tgt.driver_scd_end_year')], "inner")
              .withColumn('driver_scd_end_year', lit(None))
              .withColumn('driver_scd_status', lit(True))
              .selectExpr(
                  "src.driver_name",
                  "src.driver_nationality",
                  "src.constructor_team",
                  "src.race_year as driver_scd_start_year",
                  "driver_scd_end_year",
                  "driver_scd_status"
              )
          )

          insert_df.createOrReplaceTempView("insert_newly_modified")
          spark.sql('''
              INSERT INTO streaming_project.gold.driver_scd_details
              SELECT * FROM insert_newly_modified
          ''')
      else:
          spark.sql("""
              CREATE TABLE IF NOT EXISTS streaming_project.gold.driver_scd_details
              ( driver_name STRING,
                driver_nationality STRING,
                constructor_team STRING,
                driver_scd_start_year INT,
                driver_scd_end_year INT,
                driver_scd_status STRING
              ) USING DELTA
              PARTITIONED BY (driver_name)
          """)

      print("driver_scd_details with appending newly modified records")
      display(spark.sql('''SELECT * FROM streaming_project.gold.driver_scd_details'''))

    def process(self):
        print("Started gold-ingestion-driver_scd_details in runing....")
        race_year_list = self.read_input()  # return list of distinct race_years
        apply_tran_df = self.apply_transformations(race_year_list)  # return aggregated data

In [0]:
class Gold_race_wise_analysis():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path="streaming_project/gold"

    def __init__(self,table,race_result_max_ingestion_date,drivers_qualifying_max_ingestion_date):
         self.table=table
         #latest water mark value for race_results table from Gold_race_results class
         self.race_result_max_ingestion_date=race_result_max_ingestion_date 
         #latest water mark value for race_results table from Gold_drivers_qualifying class
         self.drivers_qualifying_max_ingestion_date=drivers_qualifying_max_ingestion_date  

    def read_input(self):
        from pyspark.sql.functions import max,col,expr,count
        race_year_list=list()
        #printing last water mark value of race_results table
        print(f"race_result_max_ingestion_date:{self.race_result_max_ingestion_date}")

        #fetching incremental race_results data from gold table race_results
        if spark.catalog.tableExists("streaming_project.gold.race_results"):
            Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
                                  .filter(col('results_ingestion_date')>self.race_result_max_ingestion_date))
            
            #fetching distinct race_year from incremental race_results data
            Incr_race_results_df_list = (Incr_race_results_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
            #fetching incremental race_results data and printing count of records
            print("race_wise_analysis:Incr_race_results_df batch count")
            display(Incr_race_results_df.select(count('*')))
            race_results_race_year_list=[r.race_year for r in Incr_race_results_df_list]

        #listing of distinct race_years of incremental race_results and printing the list
        print(f"race_year_list:{race_results_race_year_list}")


        #####################################################################
        print(f"drivers_qualifying_max_ingestion_date:{self.drivers_qualifying_max_ingestion_date}")
        #ffetching incremental drivers_qualifying data from gold table drivers_qualifying
        if spark.catalog.tableExists("streaming_project.gold.drivers_qualifying"):
            Incr_drivers_qualifying_df= (spark.read.table('streaming_project.gold.drivers_qualifying')
                                  .filter(col('qualifying_ingestion_date')>self.drivers_qualifying_max_ingestion_date))
            
            #fetching distinct race_year from incremental drivers_qualifying data
            Incr_drivers_qualifying_df_list = (Incr_drivers_qualifying_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
            #fetching incremental drivers_qualifying data and printing count of records
            print("race_wise_analysis:Incr_drivers_qualifying_df batch count")
            display(Incr_drivers_qualifying_df.select(count('*')))
            drivers_qualifying_race_year_list=[r.race_year for r in Incr_drivers_qualifying_df_list]

        #listing of distinct race_years of incremental drivers_qualifying and printing the list
        print(f"race_year_list:{drivers_qualifying_race_year_list}")
        race_year_list=[race_results_race_year_list,drivers_qualifying_race_year_list]
        return race_year_list
    
    def apply_transformations(self,race_year_list):
        from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,when,concat,lit,count,max
        from pyspark.sql.window import Window
        cc_spec_window=Window.partitionBy("race_year","driver_name").orderBy(col("race_year").asc())
        rn_spec_window=Window.partitionBy("race_year","driver_name").orderBy(col("race_name").asc())
        
        #fetching only required data from race_results table which is required for aggregation and printing the count of records
        Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
                               .filter(col('race_year').isin(race_year_list[0]))
                               )
        print("race_wise_analysis:Incr_race_results_df original count")
        display(Incr_race_results_df.select(count('*')))
        
        #find aggregated total_points for each driver in each race_year(race_results table)
        cte_df=(Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
                .agg(sum(col("result_points")).alias("total_points"))
                .select(col("race_year").alias('cte_race_year'),
                        col("driver_name").alias('cte_driver_name'),
                        col("total_points"))
                )
        # find rolling points for each driver in each race_year(race_results table)
        cte1_df=(Incr_race_results_df
                 .withColumn("rolling_points_race_year",sum(col("result_points")).over(cc_spec_window))
                 .withColumn("rolling_points_race_name",sum(col("result_points")).over(rn_spec_window))
                 .withColumn("max_fastest_lap_time",max(col('result_fastest_lap_time'))
                                                .over(Window.partitionBy(col('race_year'),col('race_name'),col('driver_name'))))
                 .select("*")
                )
        #fetching only required data from drivers_qualifying table which is required for joining and printing the count of records
        if(len(race_year_list[1]) !=0): 
            cte2_df=(spark.read.table('streaming_project.gold.drivers_qualifying')
                                .filter(col('race_year').isin(race_year_list[1]))
                                .withColumnRenamed("race_year","qualifying_race_year")
                                .withColumnRenamed("race_name","qualifying_race_name")
                                .withColumnRenamed("driver_name","qualifying_driver_name")  
                                )
        else:#if there is no incremental data in drivers_qualifying table then fetch drivers_qualifying recordes using race_years of race_results table(according to join condition it will be possible)
            cte2_df=(spark.read.table('streaming_project.gold.drivers_qualifying')
                                .filter(col('race_year').isin(race_year_list[0]))
                                .withColumnRenamed("race_year","qualifying_race_year")
                                .withColumnRenamed("race_name","qualifying_race_name")
                                .withColumnRenamed("driver_name","qualifying_driver_name")  
                                )

        print("race_wise_analysis:Incr_drivers_qualifying_df original count")
        display(cte2_df.select(count('*')))

        #aliasing the tables before join
        cte_df = cte_df.alias("cte") # aggreation applied on incremental race_results data
        cte1_df = cte1_df.alias("cte1") # transformation applied on incremental race_results data
        cte2_df=cte2_df.alias("cte2")# fetch requried columns from incremental drivers_qualifying data
        
        #performing join operation with all the tables
        join_df=(cte_df.join(cte1_df,[cte_df.cte_race_year==cte1_df.race_year,
                                      cte_df.cte_driver_name==cte1_df.driver_name],how="inner")
                        .join(cte2_df,[cte1_df.race_year==cte2_df.qualifying_race_year,
                                       cte1_df.race_name==cte2_df.qualifying_race_name,
                                      cte1_df.driver_name==cte2_df.qualifying_driver_name],how="inner")
                  .withColumn("position",dense_rank().over(Window.partitionBy(col("race_year")).orderBy(col("total_points").desc())))
                  .select([col("cte1." + c) for c in cte1_df.columns] + [col("cte.total_points"),col("position")]+[col("qualifying_q1"),col("qualifying_q2"),col("qualifying_q3")])
                  )#for fetching all columns from cte1 used different approach in select
        display(join_df.filter(col("race_year")==2018))
        return join_df
    
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approache using dynamic partitionOverwriteMode as true with overwritte mode
        (apply_tran_df.write.option("partitionOverwriteMode", "dynamic")
         .partitionBy("race_year").mode("overwrite")
         .saveAsTable(f"streaming_project.gold.{self.table}"))
        print("Data write into gold race_wise_analysis table is Done")

    def process(self):
        print("Started gold-ingestion-race_wise_analysis in runing....")
        race_year_list=self.read_input()
        apply_tran_df=self.apply_transformations(race_year_list)
        self.write_output(apply_tran_df)
                  
        


In [0]:
# from pyspark.sql.functions import col
# race_year_list=spark.read.table('streaming_project.gold.race_results').select(col('race_year')).distinct().orderBy(col('race_year').asc()).collect()
# #print(race_year_list)

# race_year_list=[r.race_year for r in race_year_list]
# print(race_year_list)

# from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,when,concat,lit
# from pyspark.sql.window import Window
# cc_spec_window=Window.partitionBy("race_year","driver_name").orderBy(col("circuit_country").asc())
# rn_spec_window=Window.partitionBy("race_year","driver_name").orderBy(col("race_name").asc())

# Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
#                         .filter(col('race_year').isin(race_year_list))
#                         )
# cte_df=(Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
#                 .agg(sum(col("result_points")).alias("total_points"))
#                 .select(col("race_year").alias('cte_race_year'),
#                         col("driver_name").alias('cte_driver_name'),
#                         col("total_points"))
#         )
# cte1_df=(Incr_race_results_df.withColumn("rolling_points_circuit_country",sum(col("result_points")).over(cc_spec_window))
#             .withColumn("rolling_points_race_name",sum(col("result_points")).over(rn_spec_window))
#             .select("*")
#         )

# cte_df = cte_df.alias("cte")
# cte1_df = cte1_df.alias("cte1")

# join_df=(cte_df.join(cte1_df,[cte_df.cte_race_year==cte1_df.race_year,
#                               cte_df.cte_driver_name==cte1_df.driver_name],how="left")
#             .withColumn("rank",dense_rank().over(Window.partitionBy("race_year").orderBy(col("total_points").desc())))
#             .select(
#             [col("cte1." + c) for c in cte1_df.columns] +
#             [col("cte.total_points"), col("rank")])
#             )
# display(join_df)

# from pyspark.sql.functions import round,col,broadcast,dense_rank,sum,when,concat,lit
# from pyspark.sql.window import Window
# spec_window=Window.partitionBy("race_year").orderBy(col('total_points').desc())
# Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
#                                .filter(col('race_year').isin(race_year_list))
#                                )
# champions_df=(Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
#                         .agg(sum(col("result_points")).alias("total_points"))
#                         .withColumn("position",dense_rank().over(spec_window))
#                         .withColumn("winning_position",when(col("position")==1,"Champion").otherwise(concat(lit("P_"),col('position'))))
#                         .select(col("race_year"),col("driver_name"),col("total_points"),col("position"),col("winning_position"))
#                     )
# display(champions_df.filter(col('driver_name')=='Lewis Hamilton'))
# from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,count,min,when
# from pyspark.sql.window import Window
# spec_window=Window.partitionBy("race_year").orderBy(col('total_points').desc())

# Incr_race_results_df= (spark.read.table('streaming_project.gold.race_results')
#                                .filter(col('race_year').isin(race_year_list))
#                                )
# season_summary_df=(Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
#                         .agg(sum(col("result_points")).alias("total_points"),
#                         count(col("race_name")).alias("grand_prix_races"),
#                         expr("count(case when result_position_order= 1 then result_position_order end)").alias("wins"),
#                         min(col("result_position_order")).alias('Best_race_results_position'),
#                     count(when(col("result_position") != 0,col("result_position"))).alias("No_of_race_finshes"),
#                     count(when(col("result_position") == 0,col("result_position"))).alias("race_does_not_finshes"),
#                     count(when(col("reslt_fastest_lap_rank") == 1, col("reslt_fastest_lap_rank"))).alias("fastest_laps"),
#                     round(avg(col("result_position_order")),2).alias("Avg_race_position")          
#                     ).withColumn("position",dense_rank().over(spec_window))
#                     .select(col("race_year"),col("driver_name"),col("position"),col("total_points"),
#                             col("grand_prix_races"),col("wins"),col("Best_race_results_position"),col("No_of_race_finshes"),col("race_does_not_finshes"),col("fastest_laps"),col("Avg_race_position"))
#                  )
# display(season_summary_df.filter(col('race_year')==2018))


In [0]:
class Gold_drivers_qualifying():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path="streaming_project/gold"

    def __init__(self,table,drivers_df,constructors_df,races_df):
         self.table=table
         self.drivers_df=drivers_df # intializing the broadcasted dataframe
         self.constructors_df=constructors_df  # intializing the broadcasted dataframe
         self.races_df=races_df  # intializing the broadcasted dataframe
    
    
    def max_watermark_value(self):
        from pyspark.sql.functions import max,col

        #fetching max_ingestion_date from gold layer table drivers_qualifying
        if spark.catalog.tableExists("streaming_project.gold.drivers_qualifying"):
           qualifying_max_ingestion_date =spark.read.table('streaming_project.gold.drivers_qualifying').agg(max(col('qualifying_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
           if qualifying_max_ingestion_date is None:
               qualifying_max_ingestion_date='1900-01-01 00:00:00'
        else:
            qualifying_max_ingestion_date='1900-01-01 00:00:00'
        #printing last water mark value 
        print(f"results_max_ingestion_date:{qualifying_max_ingestion_date}")
        return qualifying_max_ingestion_date
        
    def read_input(self,list_max_ingest):
        from pyspark.sql.functions import max,col,expr,count
        qualifying_max_ingestion_date=list_max_ingest

        #fetching incremental qualifying data and printing the count of records
        incr_qualifying_df= (spark.read.table('streaming_project.silver.qualifying')
                        .filter(col('qualifying_ingestion_date')>qualifying_max_ingestion_date)
                            )
        print("incr_qualifying_df")
        display(incr_qualifying_df.select(count("*")))

        # races_df=spark.read.table('streaming_project.silver.races')
        # constructor_df=spark.read.table('streaming_project.silver.constructors')
        # drivers_df=spark.read.table('streaming_project.silver.drivers')
        # passing all requiried tables for join as list
        read_df_list=[self.races_df,self.constructors_df,self.drivers_df,incr_qualifying_df]
        return read_df_list

    
    def apply_transformations(self,read_df_list):
        from pyspark.sql.functions import round,col,broadcast,expr
        races_df=read_df_list[0]
        constructor_df=read_df_list[1]
        drivers_df=read_df_list[2]
        qualifying_df=read_df_list[3]
        #performing join operation only on  incremental qualifying data with all other tables
        drivers_qualifying_df= (qualifying_df
                .join(races_df,qualifying_df["race_id"] == races_df["race_id"],'inner')
                .join(drivers_df,["driver_id"],'inner')
                .join(constructor_df,["constructor_id" ],'inner')
                .selectExpr("race_year","race_name","race_date","race_time","round as race_round","driver_name","driver_nationality","constructor_team","constructor_nationality","q1 as qualifying_q1"," q2 as qualifying_q2","q3 as qualifying_q3","position as qualifying_position","qualifying_ingestion_date")
                             )
        
        return drivers_qualifying_df

        
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approach using append mode
        (apply_tran_df.write.partitionBy("race_year","race_name")
         .mode("append")
         .saveAsTable(f"streaming_project.gold.{self.table}"))
        print("Data write into gold drivers_qualifying table is Done")
    
       
        
    def process(self):
        print("Started gold-ingestion-drivers_qualifying  in ran....")
        list_max_ingest=self.max_watermark_value() #return last water mark value
        read_df_list=self.read_input(list_max_ingest)#return all required tables as list 
        apply_tran_df=self.apply_transformations(read_df_list) #return joined data
        self.write_output(apply_tran_df)#write data into gold table
        return list_max_ingest #return last water mark value for other tables aggregations(season_summary,champions_list,race_wise_analysis etc...)
       
    


In [0]:
class Gold_drivers_lap_times():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path="streaming_project/gold"

    def __init__(self,table,drivers_df,races_df):
        self.table=table
        self.drivers_df=drivers_df # intializing the broadcasted dataframe
        self.races_df=races_df # intializing the broadcasted dataframe
        
    def max_watermark_value(self):
        from pyspark.sql.functions import max,col

        #fetching max_ingestion_date from gold layer table drivers_lap_times and printing that water mark value
        if spark.catalog.tableExists("streaming_project.gold.drivers_lap_times"):
            lap_times_max_ingestion_date = spark.read.table('streaming_project.gold.drivers_lap_times').agg(max(col('lap_times_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if lap_times_max_ingestion_date is None:
                lap_times_max_ingestion_date='1900-01-01 00:00:00'
        else:
            lap_times_max_ingestion_date='1900-01-01 00:00:00'
        print(f"results_max_ingestion_date:{lap_times_max_ingestion_date}")

        return lap_times_max_ingestion_date
        
    def read_input(self,list_max_ingest):
        from pyspark.sql.functions import max,col,expr,count
        lap_times_max_ingestion_date=list_max_ingest
        
        #fetching incremental lap_times data
        incr_lap_times_df= (spark.read.table('streaming_project.silver.lap_times')
                          .filter(col('lap_times_ingestion_date')>lap_times_max_ingestion_date)
                            )
                          
        print("incr_lap_times_df")
        display(incr_lap_times_df.select(count("*")))

        # races_df=spark.read.table('streaming_project.silver.races')
        # drivers_df=spark.read.table('streaming_project.silver.drivers')
        # passing all requiried tables for join as list
        read_df_list=[self.races_df,incr_lap_times_df,self.drivers_df]
        return read_df_list

    
    def apply_transformations(self,read_df_list):
        from pyspark.sql.functions import round,col,dense_rank,broadcast
        from pyspark.sql.window import Window
        races_df=read_df_list[0]
        lap_times_df=read_df_list[1]
        drivers_df=read_df_list[2]
        #performing join operation only on  incremental lap_times data with all other tables
        drivers_laptimes_df= (lap_times_df.join(races_df,['race_id'],'inner')
                                   .join(drivers_df,['driver_id'],'inner')
                                   .selectExpr("race_year","race_name","race_date","race_time","round as race_round","driver_name","driver_nationality","lap","position","time_minutes","milliseconds","lap_times_ingestion_date")
                             )
        return drivers_laptimes_df

        
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approach using append mode
        (apply_tran_df.write.partitionBy("race_year","race_name")
         .mode("append")
         .saveAsTable(f"streaming_project.gold.{self.table}"))
        print("Data write into gold drivers_lap_times table is Done")
        
    
    def process(self):
        print("Started gold-ingestion-drivers_lap_times  in ran....")
        list_max_ingest=self.max_watermark_value()# return the last water mark value
        read_df_list=self.read_input(list_max_ingest) # return the lsit of tables
        apply_tran_df=self.apply_transformations(read_df_list) # return the joined data
        self.write_output(apply_tran_df)#write data into gold table
        
        
        
    


In [0]:
class Gold_drivers_pit_stops():
    main_path="/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path="streaming_project/gold"

    def __init__(self,table,drivers_df,races_df):
        self.table=table
        self.drivers_df=drivers_df # intializing the broadcasted dataframe
        self.races_df=races_df # intializing the broadcasted dataframe
        
    def max_watermark_value(self):
        from pyspark.sql.functions import max,col

        #fetching max_ingestion_date from gold layer table drivers_pit_stops and printing that water mark value
        if spark.catalog.tableExists("streaming_project.gold.drivers_pit_stops"):
            pit_stops_max_ingestion_date =spark.read.table('streaming_project.gold.drivers_pit_stops').agg(max(col('pit_stops_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if pit_stops_max_ingestion_date is None:
                pit_stops_max_ingestion_date='1900-01-01 00:00:00'
        else:
            pit_stops_max_ingestion_date='1900-01-01 00:00:00'
        print(f"results_max_ingestion_date:{pit_stops_max_ingestion_date}")

        return pit_stops_max_ingestion_date

    def read_input(self,list_max_ingest):
        from pyspark.sql.functions import max,col,expr,count
        #fetching incremental pit_stops data
        pit_stops_max_ingestion_date=list_max_ingest
        incr_pit_stops_df= (spark.read.table('streaming_project.silver.pit_stops')
                        .filter(col('pit_stops_ingestion_date')>pit_stops_max_ingestion_date)
                            )
                          
        print("incr_pit_stops_df")
        display(incr_pit_stops_df.select(count("*")))
        # races_df=spark.read.table('streaming_project.silver.races')
        # drivers_df=spark.read.table('streaming_project.silver.drivers')

        # passing all requiried tables for join as list
        read_df_list=[self.races_df,incr_pit_stops_df,self.drivers_df]
        return read_df_list

    
    def apply_transformations(self,read_df_list):
        from pyspark.sql.functions import round,col,dense_rank,broadcast
        from pyspark.sql.window import Window
        races_df=read_df_list[0]
        pit_stops_df=read_df_list[1]
        drivers_df=read_df_list[2]
        #performing join operation only on  incremental pit_stops data with all other tables
        drivers_pit_stops_df= (pit_stops_df.join(races_df,['race_id'],'inner')
                                   .join(drivers_df,['driver_id'],'inner')
                                   .selectExpr("race_year","race_name","race_date","race_time","round as race_round","driver_name","driver_nationality","stop","lap","time","duration_sec","duration_minutes","milliseconds","pit_stops_ingestion_date")
                             )
        return drivers_pit_stops_df

        
    def write_output(self,apply_tran_df):
        # writing those data into gold layer table by partitioning according to filter approach using append mode
        (apply_tran_df.write.partitionBy("race_year","race_name")
         .mode("append")
         .saveAsTable(f"streaming_project.gold.{self.table}"))
        print("Data write into gold drivers_pit_stops table is Done")
        
    
       
        
    def process(self):
        print("Started gold-ingestion-drivers_pit_stops  in ran....")
        list_max_ingest=self.max_watermark_value() # return last water mark value
        read_df_list=self.read_input(list_max_ingest) # return the lsit of tables
        apply_tran_df=self.apply_transformations(read_df_list) # return the joined data
        self.write_output(apply_tran_df)#write data into gold table


In [0]:
#  Enable dynamic partition overwrite
# spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")